# Five acoustic CNN architectures

Each network takes a single 128 × 313 log-mel clip and returns four machine-ID logits for a known machine type. This walkthrough constructs the layer layouts with random weights and checks output shapes. The reported fits use healthy machine-ID training and the initialization described below.

In [1]:
from pathlib import Path
import os
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebooks/code").is_dir() and (p / "results/fits.json").is_file())
os.chdir(root)
get_ipython().run_line_magic("run", '"notebooks/00_runtime.ipynb"')
import torch
from mimii.mobilenet import build_mobilenet
from mimii.transfer_models import build_transfer
from mimii.acoustic_cnn import AcousticResidualStatsCNN
from mimii.large_acoustic_cnn import LargeAcousticResidualStatsCNN
torch.set_num_threads(2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Example forward pass on", device)

Example forward pass on cuda


## ImageNet backbones

The trained models start from ImageNet weights, combine the three input-channel filters into a mono stem, and replace the final layer with a four-ID classifier.

In [2]:
models = {"MobileNetV3-Small": build_mobilenet(),
          "ResNet-18": build_transfer("resnet18", pretrained=False),
          "EfficientNet-B0": build_transfer("efficientnet_b0", pretrained=False)}

## Custom acoustic networks

The custom residual blocks use depthwise convolutions and channel attention. The pooling layer retains four frequency regions and measures temporal mean and variability. The wider model changes channel and classifier widths.

In [3]:
models.update({"Compact CNN": AcousticResidualStatsCNN(),
               "Wide CNN": LargeAcousticResidualStatsCNN()})

## What changes when the acoustic CNN is widened?

Count convolution and linear modules directly, including pointwise channel gates and shortcut projections. ResNet-18’s name counts its main path. The compact and wide networks keep six residual blocks each. Every forward pass must return four logits.

In [4]:
from IPython.display import Markdown, display

clip = torch.zeros(1, 1, 128, 313, device=device)
lines = ["| Architecture | Convolutions | Dense layers | Parameters per type | Output |",
         "|---|---:|---:|---:|---|"]
with torch.inference_mode():
    for name, model in models.items():
        logits = model.eval().to(device)(clip)
        assert logits.shape == (1, 4)
        conv = sum(isinstance(layer, torch.nn.Conv2d) for layer in model.modules())
        dense = sum(isinstance(layer, torch.nn.Linear) for layer in model.modules())
        params = sum(p.numel() for p in model.parameters())
        lines.append(f"| {name} | {conv} | {dense} | {params:,} | 1 × 4 |")
display(Markdown("\n".join(lines)))

| Architecture | Convolutions | Dense layers | Parameters per type | Output |
|---|---:|---:|---:|---|
| MobileNetV3-Small | 52 | 2 | 1,521,668 | 1 × 4 |
| ResNet-18 | 20 | 1 | 11,172,292 | 1 × 4 |
| EfficientNet-B0 | 81 | 1 | 4,012,096 | 1 × 4 |
| Compact CNN | 34 | 2 | 177,724 | 1 × 4 |
| Wide CNN | 34 | 2 | 1,286,964 | 1 × 4 |